In [1]:
import numpy as np
import pandas as pd

This script creates a csv of vaccination rates by school based on [data](https://dph.sc.gov/sites/scdph/files/2026-03/45_Day_Report_25-26_20260310.pdf) from the [NC Division of Public Health](https://www.dph.ncdhhs.gov/programs/epidemiology/communicable-disease/infectious-respiratory-diseases/measles-rubeola/nc-measles-vaccination-data-dashboard). The first link provides the data in pdf format. This repo provides the data in Excel format in the data folder.

In [2]:
# NC vax rates: https://www.dph.ncdhhs.gov/programs/epidemiology/communicable-disease/infectious-respiratory-diseases/measles-rubeola/nc-measles-vaccination-data-dashboard

# Read in vax rates by school
vax_rates = pd.read_excel('data/SC_School_Vax_Rates.xlsx') # https://dph.sc.gov/sites/scdph/files/2026-03/45_Day_Report_25-26_20260310.pdf
# Filter for Spartanburg County and public schools
vax_rates = vax_rates.loc[(vax_rates['County'] == 'Spartanbur') & (vax_rates['Type'] == 'Public')]
# Reformat vax rates to be floats
vax_rates.loc[vax_rates["Immunizations"] == ">96%", "Immunizations"] = '0.96'
vax_rates["Immunizations"] = vax_rates["Immunizations"].astype(float)
vax_rates = vax_rates[['School Name', 'Immunizations']].rename(columns={'School Name': 'NAME', 'Immunizations': 'vax_rate'})
vax_rates

,NAME,vax_rate
1372,Meeting Street Academy-Spartanburg,0.96
1373,Woodruff Elementary,0.96
1374,Clifdale Middle,0.96
1375,Woodruff High,0.96
1376,Mary H. Wright Elementary,0.96
...,...,...
1457,SC School for the Deaf High,0.76
1458,Carlisle-Foster's Grove Elementary,0.75
1459,Boiling Springs Middle,0.75
1460,Libertas Academy - Boiling Springs,0.74


In [3]:
# Read in list of public schools and filter for Spartanburg County 
public_schools = pd.read_excel('data/school/EDGE_GEOCODE_PUBLICSCH_1920.xlsx') # big file, takes a while
public_schools = public_schools.loc[(public_schools['CNTY'] == 45083)] # filter for Spartanburg County
public_schools = public_schools[['NCESSCH', 'NAME']]
public_schools

,NCESSCH,NAME
77691,450000100255,Tyger River High
77706,450000401226,SC School for the Blind High
77707,450000401227,SC School for the Deaf High
77708,450000401228,SC School for the Deaf Elementary/Middle
77709,450000401246,SC School for the Blind Elementary/Middle
...,...,...
78763,450366001664,Meeting Street Academy-Spartanburg
78764,450366001710,Drayton Mills Elementary
78855,450390101541,Spartanburg Preparatory School
78869,450390101639,High Point Academy


In [4]:
# Map vax rates onto schools in our population
sch_vax_rates = public_schools.merge(vax_rates, on='NAME', how='left')
# Identify schools in vax_rates but not in public_schools
sch_vax_rates.loc[sch_vax_rates['vax_rate'].isnull()]
# Assign these schools to have vax_rate=0.92
sch_vax_rates.loc[sch_vax_rates['vax_rate'].isnull(), 'vax_rate'] = 0.92
# Rename NCESSCH to sch_code
sch_vax_rates = sch_vax_rates.rename(columns={'NCESSCH': 'sch_code'})
# Now all schools have vax_rate
# sch_vax_rates = sch_vax_rates[['sch_code', 'vax_rate']]
sch_vax_rates.to_csv('data/sch_vax_rates.csv', index=False)
sch_vax_rates

,sch_code,NAME,vax_rate
0,450000100255,Tyger River High,0.92
1,450000401226,SC School for the Blind High,0.92
2,450000401227,SC School for the Deaf High,0.76
3,450000401228,SC School for the Deaf Elementary/Middle,0.84
4,450000401246,SC School for the Blind Elementary/Middle,0.95
...,...,...,...
80,450366001664,Meeting Street Academy-Spartanburg,0.96
81,450366001710,Drayton Mills Elementary,0.94
82,450390101541,Spartanburg Preparatory School,0.94
83,450390101639,High Point Academy,0.87
